In [8]:
 !pip install openai python-dotenv -q

import openai
import json
import os

# Попытка получить ключ из Colab Secrets (если запуск в Colab)
try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
except:
    # Иначе из переменных окружения (если запуск локально)
    from dotenv import load_dotenv
    load_dotenv()
    OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "your-key-here")

client = openai.OpenAI(api_key=OPENAI_API_KEY)

In [9]:
SYSTEM_PROMPT = """
Ты — AI-агент для обработки обращений клиентов. Твоя задача:
1. Определить тип запроса: жалоба, вопрос, предложение, срочное.
2. Кратко суммировать обращение (1-2 предложения).
3. Определить срочность: низкая, средняя, высокая.

Отвечай ТОЛЬКО в формате JSON, без лишнего текста:
{
  "type": "тип запроса",
  "summary": "краткое саммари",
  "urgency": "срочность",
  "original_text": "исходный текст"
}
"""

In [10]:
def process_request(user_text):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_text}
        ],
        temperature=0.1
    )
    result = response.choices[0].message.content
    return json.loads(result)

In [11]:
test_text = "Здравствуйте, я уже неделю не могу дозвониться до поддержки. Мой заказ №4567 до сих пор не отправлен. Это безобразие!"

result = process_request(test_text)
print(json.dumps(result, ensure_ascii=False, indent=2))

{
  "type": "жалоба",
  "summary": "Клиент жалуется на невозможность дозвониться до поддержки и задержку с отправкой заказа.",
  "urgency": "высокая",
  "original_text": "Здравствуйте, я уже неделю не могу дозвониться до поддержки. Мой заказ №4567 до сих пор не отправлен. Это безобразие!"
}


In [12]:
{
  "type": "жалоба",
  "summary": "Клиент не может дозвониться до поддержки, заказ №4567 не отправлен",
  "urgency": "высокая",
  "original_text": "Здравствуйте, я уже неделю не могу дозвониться до поддержки..."
}

{'type': 'жалоба',
 'summary': 'Клиент не может дозвониться до поддержки, заказ №4567 не отправлен',
 'urgency': 'высокая',
 'original_text': 'Здравствуйте, я уже неделю не могу дозвониться до поддержки...'}